In [ ]:
!pip install -q datasets transformers evaluate scikit-learn matplotlib seaborn accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.2 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

#Загрузка датасета
data = load_dataset("dair-ai/emotion", trust_remote_code=True)

train_texts = data['train']['text']
train_labels = data['train']['label']
val_texts = data['validation']['text']
val_labels = data['validation']['label']

results_table = []

def add_result(model_name, classifier_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    results_table.append({
        "Модель": model_name,
        "Классификатор": classifier_name,
        "Accuracy": acc,
        "Macro-F1": f1
    })
    print(f"[{model_name} | {classifier_name}] Accuracy: {acc:.4f}, Macro-F1: {f1:.4f}")

def get_cls_embeddings(texts, model_name, batch_size=128):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            #Берем вектор первого токена [CLS]
            cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_emb)

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    return np.vstack(all_embeddings)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'dair-ai/emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'dair-ai/emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Используемое устройство: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
print("Feature Extraction без Fine-Tuning")

base_models = ["bert-base-uncased", "distilbert-base-uncased"]

for model_name in base_models:
    print(f"\nИзвлечение эмбеддингов для {model_name}...")
    X_train = get_cls_embeddings(train_texts, model_name)
    X_val = get_cls_embeddings(val_texts, model_name)

    #Logistic Regression
    clf_lr = LogisticRegression(max_iter=1000)
    clf_lr.fit(X_train, train_labels)
    preds_lr = clf_lr.predict(X_val)
    add_result(model_name, "Logistic Regression", val_labels, preds_lr)

    #SVM (LinearSVC для скорости)
    clf_svm = LinearSVC(max_iter=2000, dual=False)
    clf_svm.fit(X_train, train_labels)
    preds_svm = clf_svm.predict(X_val)
    add_result(model_name, "SVM (Linear)", val_labels, preds_svm)

Feature Extraction без Fine-Tuning

Извлечение эмбеддингов для bert-base-uncased...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[distilbert-base-uncased | Logistic Regression] Accuracy: 0.6330, Macro-F1: 0.5298
[distilbert-base-uncased | SVM (Linear)] Accuracy: 0.6295, Macro-F1: 0.5177


In [ ]:
print("\nГотовые Fine-Tuned модели")

finetuned_models = [
    "bhadresh-savani/bert-base-uncased-emotion",
    "Panda0116/emotion-classification-model" # DistilBERT
]

for model_name in finetuned_models:
    print(f"\nРабота с {model_name}")

    classifier = pipeline("text-classification", model=model_name, device=0, truncation=True, max_length=512)

    model_label2id = {k.lower(): v for k, v in classifier.model.config.label2id.items()}

    for i in range(6):
        model_label2id[f"label_{i}"] = i

    print("Получение предсказаний встроенного классификатора")
    pipe_preds = classifier(list(val_texts), batch_size=128)

    preds_pipe = [model_label2id[p['label'].lower()] for p in pipe_preds]
    add_result(model_name, "Встроенный (Pipeline)", val_labels, preds_pipe)

    del classifier
    torch.cuda.empty_cache()

    print("Извлечение CLS эмбеддингов из fine-tuned модели")
    X_train = get_cls_embeddings(train_texts, model_name)
    X_val = get_cls_embeddings(val_texts, model_name)

    clf_svm = LinearSVC(max_iter=2000, dual=False)
    clf_svm.fit(X_train, train_labels)
    preds_svm = clf_svm.predict(X_val)
    add_result(f"{model_name} (CLS)", "SVM (Linear)", val_labels, preds_svm)


Готовые Fine-Tuned модели

Работа с bhadresh-savani/bert-base-uncased-emotion


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bhadresh-savani/bert-base-uncased-emotion
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Получение предсказаний встроенного классификатора
[bhadresh-savani/bert-base-uncased-emotion | Встроенный (Pipeline)] Accuracy: 0.9405, Macro-F1: 0.9198
Извлечение CLS эмбеддингов из fine-tuned модели


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: Panda0116/emotion-classification-model
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.weight     | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: Panda0116/emotion-classification-model
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.weight     | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Panda0116/emotion-classification-model (CLS) | SVM (Linear)] Accuracy: 0.9345, Macro-F1: 0.9093


In [ ]:
print("\nСобственный Fine-Tuning")
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

ft_model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(ft_model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = data.map(tokenize_function, batched=True)

model_ft = AutoModelForSequenceClassification.from_pretrained(ft_model_name, num_labels=6).to(device)

training_args = TrainingArguments(
    output_dir="./emotion_results",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3, #3 эпохи
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True, #Mixed Precision для ускорения на GPU
    seed=42,
    logging_steps=100
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro")
    }

trainer = Trainer(
    model=model_ft,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()

print("Оценка после Fine-Tuning")
eval_results = trainer.evaluate()
add_result("Мой Fine-Tuned BERT", "Встроенный", val_labels, np.argmax(trainer.predict(tokenized_datasets["validation"]).predictions, axis=-1))

trainer.save_model("./my_best_emotion_model")
tokenizer.save_pretrained("./my_best_emotion_model")

del model_ft, trainer
torch.cuda.empty_cache()
gc.collect()

print("Извлечение CLS из нашей дообученной модели")
X_train_my = get_cls_embeddings(train_texts, "./my_best_emotion_model")
X_val_my = get_cls_embeddings(val_texts, "./my_best_emotion_model")

clf_svm_my = LinearSVC(max_iter=2000, dual=False)
clf_svm_my.fit(X_train_my, train_labels)
preds_svm_my = clf_svm_my.predict(X_val_my)
add_result("Мой Fine-Tuned BERT (CLS)", "SVM (Linear)", val_labels, preds_svm_my)


Собственный Fine-Tuning


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.249190,0.216294,0.919500,0.894474
2,0.160190,0.154693,0.940000,0.914758
3,0.108198,0.150316,0.938000,0.914807


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./my_best_emotion_model
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./my_best_emotion_model
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Мой Fine-Tuned BERT (CLS) | SVM (Linear)] Accuracy: 0.9350, Macro-F1: 0.9092


In [ ]:
print("\nИтоговая таблица")

df_results = pd.DataFrame(results_table)
display(df_results.sort_values(by="Accuracy", ascending=False).reset_index(drop=True))


Итоговая таблица


,Модель,Классификатор,Accuracy,Macro-F1
0,Panda0116/emotion-classification-model,Встроенный (Pipeline),0.9425,0.919177
1,bhadresh-savani/bert-base-uncased-emotion,Встроенный (Pipeline),0.9405,0.919810
2,Мой Fine-Tuned BERT,Встроенный,0.9380,0.914807
3,bhadresh-savani/bert-base-uncased-emotion (CLS),SVM (Linear),0.9380,0.916653
4,Мой Fine-Tuned BERT (CLS),SVM (Linear),0.9350,0.909204
5,Panda0116/emotion-classification-model (CLS),SVM (Linear),0.9345,0.909264
6,distilbert-base-uncased,Logistic Regression,0.6330,0.529805
7,distilbert-base-uncased,SVM (Linear),0.6295,0.517731
8,bert-base-uncased,SVM (Linear),0.6025,0.490813
9,bert-base-uncased,Logistic Regression,0.5950,0.493884
